# MedSafe expanded-coverage training
This reproduces the additional models used beyond the original 300 detailed-mechanism records. It does not replace the original Colab text model.

Reference coverage is 7,545 pairs, but supervised data remains 300 research-labeled rows: 192 training, 48 validation, 60 test. Do not claim 7,545 labeled training examples or clinical validation. These labels are AI-assisted. The existing test partition has been inspected previously, so it is not a fresh external test.

## 1. Install matching packages
Use a Python 3.12 or 3.13 Colab runtime. Restart the runtime if Colab requests it after installation.


In [ ]:
%pip -q install numpy==2.1.3 pandas==2.2.3 scikit-learn==1.6.1 joblib==1.6.0


## 2. Upload five files from your GitHub checkout
- `MedSafe/training/train_coverage_model.py`
- `MedSafe/backend/feature_schema.py`
- `MedSafe/backend/data/reference_catalog.json`
- `MedSafe/backend/data/medsafe_suggestions_300.csv`
- `MedSafe/training/results/split_manifest.csv`

The catalog contains severity and source classifications only. Pair names, source URLs, target labels, and management advice are excluded from inference features.


In [ ]:
from google.colab import files
from pathlib import Path
import json
uploaded = files.upload()
required = {'train_coverage_model.py', 'feature_schema.py', 'reference_catalog.json', 'medsafe_suggestions_300.csv', 'split_manifest.csv'}
missing = required - set(uploaded)
if missing:
    raise ValueError(f'Missing files: {sorted(missing)}')


## 3. Train and evaluate
The script uses DictVectorizer plus three one-vs-rest Logistic Regression classifiers per model. It trains one severity-plus-classification model and one severity-only model. It fits only the original training partition, selects C from 0.1, 1, 10 using validation macro-F1, then evaluates on the fixed test partition. It also reports the majority baseline. No predictions are used as new labels.


In [ ]:
%run train_coverage_model.py --catalog reference_catalog.json --training-csv medsafe_suggestions_300.csv --split-manifest split_manifest.csv --output coverage_export


## 4. Inspect limitations and label-level results
These are component metrics, not full-system performance or medical validation. A new independent evaluation is still needed, especially for pairs outside the labeled subset and for severity-only predictions.


In [ ]:
evaluation = json.loads(Path('coverage_export/coverage_evaluation.json').read_text())
print(json.dumps(evaluation, indent=2))


## 5. Download separate artifacts
Copy `coverage_suggestions.joblib` and `coverage_metadata.json` into `MedSafe/backend/models/`. Copy the evaluation and held-out prediction JSON files into `MedSafe/training/results/`. Keep `reference_catalog.json` matched to its exported checksum. Preserve the original `medsafe_suggestions.joblib`, `metadata.json`, and original evaluation files. The requirements file records the versions used; keep them consistent with the original model before integrating the exports. Restart the server after replacing artifacts.


In [ ]:
for name in ['coverage_suggestions.joblib', 'coverage_metadata.json', 'coverage_evaluation.json', 'coverage_heldout_predictions.json', 'requirements.txt']:
    files.download(str(Path('coverage_export') / name))
